In [5]:
import psycopg2

# Test basic connection
try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="marketing_attribution",
        user="postgres",
        password="NewStrongPassword@123"  # Replace with your password
    )
    print("✅ Connection successful!")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed!")
    print(f"Error: {e}")

✅ Connection successful!


In [8]:
# Load CSV Data into PostgreSQL Database

import pandas as pd
import psycopg2
from sqlalchemy import create_engine
import numpy as np
from urllib.parse import quote_plus

print("Starting data load process...")

# ============================================
# DATABASE CONNECTION CONFIGURATION
# ============================================

# IMPORTANT: Replace with your actual password
DB_PASSWORD = 'NewStrongPassword@123'  # ⚠️ PUT YOUR PASSWORD HERE

DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'marketing_attribution',
    'user': 'postgres',
    'password': DB_PASSWORD
}

print("Connecting to PostgreSQL database...")

# URL-encode the password to handle special characters
encoded_password = quote_plus(DB_CONFIG['password'])

# Create SQLAlchemy engine with properly encoded password
connection_string = f"postgresql://{DB_CONFIG['user']}:{encoded_password}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

engine = create_engine(connection_string)

print("✅ Database connection established!")

# ============================================
# LOAD CSV FILES
# ============================================

print("\nLoading CSV files...")

# Load customer journeys
df_journeys = pd.read_csv("D:/Projects/End-to-end projects/5. Marketing Attribution Project/Data/customer_journeys.csv")
df_journeys['touchpoint_date'] = pd.to_datetime(df_journeys['touchpoint_date'])
print(f"✅ Loaded customer_journeys.csv: {len(df_journeys):,} rows")

# Load conversions
df_conversions = pd.read_csv("D:/Projects/End-to-end projects/5. Marketing Attribution Project/Data/conversions.csv")
df_conversions['conversion_date'] = pd.to_datetime(df_conversions['conversion_date'])
print(f"✅ Loaded conversions.csv: {len(df_conversions):,} rows")

# Load marketing spend
df_spend = pd.read_csv("D:/Projects/End-to-end projects/5. Marketing Attribution Project/Data/marketing_spend.csv")
df_spend['date'] = pd.to_datetime(df_spend['date'])
print(f"✅ Loaded marketing_spend.csv: {len(df_spend):,} rows")

# ============================================
# INSERT DATA INTO POSTGRESQL TABLES
# ============================================

print("\nInserting data into PostgreSQL tables...")

# Insert customer journeys
print("Inserting customer journeys...")
df_journeys.to_sql(
    'customer_journeys', 
    engine, 
    if_exists='append',  # append to existing table
    index=False,
    method='multi',
    chunksize=1000
)
print(f"✅ Inserted {len(df_journeys):,} rows into customer_journeys table")

# Insert conversions
print("Inserting conversions...")
df_conversions.to_sql(
    'conversions', 
    engine, 
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)
print(f"✅ Inserted {len(df_conversions):,} rows into conversions table")

# Insert marketing spend
print("Inserting marketing spend...")
df_spend.to_sql(
    'marketing_spend', 
    engine, 
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)
print(f"✅ Inserted {len(df_spend):,} rows into marketing_spend table")

# ============================================
# VERIFY DATA LOADED CORRECTLY
# ============================================

print("\n" + "="*60)
print("VERIFYING DATA IN DATABASE")
print("="*60)

# Test queries
query_journeys = "SELECT COUNT(*) as count FROM customer_journeys"
query_conversions = "SELECT COUNT(*) as count FROM conversions"
query_spend = "SELECT COUNT(*) as count FROM marketing_spend"

count_journeys = pd.read_sql(query_journeys, engine).iloc[0]['count']
count_conversions = pd.read_sql(query_conversions, engine).iloc[0]['count']
count_spend = pd.read_sql(query_spend, engine).iloc[0]['count']

print(f"\n📊 customer_journeys table: {count_journeys:,} rows")
print(f"📊 conversions table: {count_conversions:,} rows")
print(f"📊 marketing_spend table: {count_spend:,} rows")

# Sample data from each table
print("\n" + "="*60)
print("SAMPLE DATA FROM TABLES")
print("="*60)

print("\n🔍 Sample from customer_journeys:")
sample_journeys = pd.read_sql("SELECT * FROM customer_journeys LIMIT 5", engine)
print(sample_journeys)

print("\n🔍 Sample from conversions:")
sample_conversions = pd.read_sql("SELECT * FROM conversions LIMIT 5", engine)
print(sample_conversions)

print("\n🔍 Sample from marketing_spend:")
sample_spend = pd.read_sql("SELECT * FROM marketing_spend LIMIT 5", engine)
print(sample_spend)

# ============================================
# QUICK DATA QUALITY CHECKS
# ============================================

print("\n" + "="*60)
print("DATA QUALITY CHECKS")
print("="*60)

# Check for null values
null_check = """
SELECT 
    'customer_journeys' as table_name,
    COUNT(*) FILTER (WHERE customer_id IS NULL) as null_customer_id,
    COUNT(*) FILTER (WHERE channel IS NULL) as null_channel
FROM customer_journeys
UNION ALL
SELECT 
    'conversions' as table_name,
    COUNT(*) FILTER (WHERE customer_id IS NULL) as null_customer_id,
    COUNT(*) FILTER (WHERE revenue IS NULL) as null_revenue
FROM conversions;
"""

null_results = pd.read_sql(null_check, engine)
print("\n✅ Null Value Check:")
print(null_results)

# Check date ranges
date_check = """
SELECT 
    MIN(touchpoint_date) as min_date,
    MAX(touchpoint_date) as max_date,
    COUNT(DISTINCT customer_id) as unique_customers
FROM customer_journeys;
"""

date_results = pd.read_sql(date_check, engine)
print("\n✅ Date Range Check:")
print(date_results)

# Channel distribution
channel_dist = """
SELECT 
    channel,
    COUNT(*) as touchpoint_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM customer_journeys
GROUP BY channel
ORDER BY touchpoint_count DESC;
"""

channel_results = pd.read_sql(channel_dist, engine)
print("\n✅ Channel Distribution:")
print(channel_results)

print("\n" + "="*60)
print("✅ DATA LOAD COMPLETE!")
print("="*60)
print("\nAll data has been successfully loaded into PostgreSQL.")
print("You can now run SQL queries in pgAdmin or continue with Python analysis.")

# Close connection
engine.dispose()

Starting data load process...
Connecting to PostgreSQL database...
✅ Database connection established!

Loading CSV files...
✅ Loaded customer_journeys.csv: 29,635 rows
✅ Loaded conversions.csv: 1,511 rows
✅ Loaded marketing_spend.csv: 1,104 rows

Inserting data into PostgreSQL tables...
Inserting customer journeys...
✅ Inserted 29,635 rows into customer_journeys table
Inserting conversions...
✅ Inserted 1,511 rows into conversions table
Inserting marketing spend...
✅ Inserted 1,104 rows into marketing_spend table

VERIFYING DATA IN DATABASE

📊 customer_journeys table: 29,635 rows
📊 conversions table: 1,511 rows
📊 marketing_spend table: 1,104 rows

SAMPLE DATA FROM TABLES

🔍 Sample from customer_journeys:
   journey_id  customer_id  touchpoint_number touchpoint_date       channel  \
0           1            1                  1      2024-09-16   Paid_Search   
1           2            2                  1      2024-07-09  Facebook_Ads   
2           3            2                  2    